# Anticipation in Football: From Predicting Opponents to Understanding Teammates
### MIT Sloan Sports Analytics Conference (SSAC 2027) Research Abstract Reproduction Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fady-nasser/anticipation/blob/main/notebooks/reproduce_anticipation.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repository-blue.svg)](https://github.com/fady-nasser/anticipation)
[![Code license: MIT](https://img.shields.io/badge/Code%20license-MIT-blue.svg)](https://github.com/fady-nasser/anticipation/blob/main/LICENSE)

This notebook **reproduces all anticipation event counts and figures from scratch** directly from the extracted 5 Hz velocity profiles (`.npz` files).
* **Zero precomputed CSV counts loaded:** Every dyadic interaction is recalculated via Granger vector autoregression.
* **Continuous ball motion control:** Controls for ball dynamics in every window to remove chasing bias.
* **Privacy compliant:** Uses 5 Hz directional velocity profiles ($v_x, v_y$) rather than proprietary raw pitch coordinates.


## 1. Environment Setup & Data Acquisition
Clone the repository to acquire the extracted feature `.npz` files and install minimal requirements.


In [ ]:
# Clone repository if running in Google Colab
import os, sys

if not os.path.exists('data/extracted_features'):
    print('Cloning repository from GitHub...')
    !git clone https://github.com/fady-nasser/anticipation.git
    %cd anticipation

!pip install -q numpy pandas scipy matplotlib seaborn
print('Environment and data ready!')


## 2. Imports and Configuration


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.anticipation.pipeline import run_window_anticipation_counts

print('Anticipation pipeline ready!')


## 3. Inspecting Extracted Velocity Features
Let's inspect the 5 Hz velocity arrays from the World Cup Final archive (`match_10517_final_features.npz`):


In [ ]:
data_final = np.load('data/extracted_features/match_10517_final_features.npz')
print('Extracted arrays in Match 10517:', len(data_final.files))
print('Sample arrays:', data_final.files[:10])

# Preview of 5 Hz velocity components
preview_df = pd.DataFrame({
    'Period': data_final['period'][:8],
    'Time (s)': np.round(data_final['t'][:8], 2),
    'Ball vx (m/s)': np.round(data_final['ball_vx'][:8], 2),
    'Ball vy (m/s)': np.round(data_final['ball_vy'][:8], 2),
    'Messi vx (ARG_10)': np.round(data_final['ARG_10_vx'][:8], 2),
    'Messi vy (ARG_10)': np.round(data_final['ARG_10_vy'][:8], 2),
    'Mbappe vx (FRA_10)': np.round(data_final['FRA_10_vx'][:8], 2),
    'Mbappe vy (FRA_10)': np.round(data_final['FRA_10_vy'][:8], 2),
})
preview_df


## 4. Recalculating Figure 1 Directly from Extracted Features (.npz)
We now run the Granger causality loop directly across all 15-second rolling windows for France, Argentina, Morocco, and Serbia.


In [ ]:
print('Recalculating within-team anticipation counts directly from .npz archives...\n')
t0 = time.time()

# 1. France (Final vs Argentina)
fra_roster = {
    'FRA_4':'Varane', 'FRA_5':'Kounde', 'FRA_7':'Griezmann', 'FRA_8':'Tchouameni',
    'FRA_9':'Giroud', 'FRA_10':'Mbappe', 'FRA_11':'Dembele', 'FRA_12':'KoloMuani',
    'FRA_14':'Rabiot', 'FRA_18':'Upamecano', 'FRA_20':'Coman', 'FRA_22':'THernandez',
    'FRA_25':'Camavinga', 'FRA_26':'M.Thuram'
}
fra_counts = run_window_anticipation_counts(
    'data/extracted_features/match_10517_final_features.npz',
    fra_roster, fra_roster, outliers=['Kounde']
)
np.fill_diagonal(fra_counts.values, 0)
print(f'Done France in {time.time()-t0:.1f}s')

# 2. Argentina (Semi-Final 3-0 vs Croatia)
t_team = time.time()
arg_roster = {
    'ARG_3':'Tagliafico', 'ARG_5':'Paredes', 'ARG_7':'De Paul', 'ARG_9':'Alvarez',
    'ARG_10':'Messi', 'ARG_13':'Romero', 'ARG_14':'Palacios', 'ARG_19':'Otamendi',
    'ARG_20':'Mac Allister', 'ARG_21':'Dybala', 'ARG_24':'Enzo', 'ARG_25':'L.Martinez', 'ARG_26':'Molina'
}
arg_counts = run_window_anticipation_counts(
    'data/extracted_features/match_10514_argentina_features.npz',
    arg_roster, arg_roster, outliers=['Paredes', 'Enzo']
)
np.fill_diagonal(arg_counts.values, 0)
print(f'Done Argentina in {time.time()-t_team:.1f}s')

# 3. Morocco (Semi-Final vs France)
t_team = time.time()
mar_roster = {
    'MAR_2':'Hakimi', 'MAR_3':'Mazraoui', 'MAR_4':'Amrabat', 'MAR_7':'Ziyech',
    'MAR_8':'Ounahi', 'MAR_9':'Hamdallah', 'MAR_14':'Aboukhlal', 'MAR_15':'Amallah',
    'MAR_17':'Boufal', 'MAR_18':'El Yamiq', 'MAR_19':'En-Nesyri', 'MAR_20':'Dari', 'MAR_25':'Attiyat Allah'
}
mar_counts = run_window_anticipation_counts(
    'data/extracted_features/match_10515_morocco_features.npz',
    mar_roster, mar_roster, outliers=['Amrabat']
)
np.fill_diagonal(mar_counts.values, 0)
print(f'Done Morocco in {time.time()-t_team:.1f}s')

# 4. Serbia (Group Stage 3-3 vs Cameroon)
t_team = time.time()
srb_roster = {
    'SRB_2':'Pavlovic', 'SRB_4':'Milenkovic', 'SRB_5':'Veljkovic', 'SRB_6':'Maksimovic',
    'SRB_7':'Radonjic', 'SRB_9':'A.Mitrovic', 'SRB_10':'Tadic', 'SRB_13':'S.Mitrovic',
    'SRB_14':'Zivkovic', 'SRB_16':'Lukic', 'SRB_17':'Kostic', 'SRB_20':'S.Milinkovic',
    'SRB_21':'Djuricic', 'SRB_26':'Grujic'
}
srb_counts = run_window_anticipation_counts(
    'data/extracted_features/match_3840_serbia_features.npz',
    srb_roster, srb_roster, outliers=[]
)
np.fill_diagonal(srb_counts.values, 0)
print(f'Done Serbia in {time.time()-t_team:.1f}s')

print(f'\nAll 4 teams recalculated from scratch in {time.time()-t0:.1f}s!')


## 5. Render Figure 1: Within-Team Anticipation Maps
Plotting the freshly calculated matrices with explicit integer values and no blank masks:


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 20))
fig.suptitle('FIFA World Cup 2022 : Cumulative Anticipation Event Counts (15s Windows)\n'
             'Total Discrete Tactical Phases Where Leader Triggered Follower (Controlling for Ball Motion)',
             fontsize=16, fontweight='bold', y=0.98)

teams = [
    ('(A) France (2022 WC Final vs Argentina)', fra_counts),
    ('(B) Argentina (2022 WC Semi-Final 3-0 vs Croatia)', arg_counts),
    ('(C) Morocco (2022 WC Semi-Final vs France)', mar_counts),
    ('(D) Serbia (2022 WC Group Stage 3-3 vs Cameroon)', srb_counts),
]
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for idx, (title, cdf) in enumerate(teams):
    r, c = positions[idx]
    ax = axes[r, c]
    sns.heatmap(cdf, ax=ax, cmap='Blues', annot=True, fmt='d', linewidths=0.4,
                mask=None, cbar_kws={'label': 'Anticipation Event Count'}, vmin=0)
    flat = cdf.stack()
    top_pair = flat.idxmax()
    top_val = flat.max()
    ax.set_title(f"{title}\nTop Pair: {top_pair[0]} -> {top_pair[1]} ({top_val} times)", fontsize=12, fontweight='bold')
    ax.set_xlabel('Follower (Reactor)', fontsize=10)
    ax.set_ylabel('Leader (Trigger)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 6. Recalculating Figure 2 Directly from Extracted Features (.npz)
We now recalculate cross-team anticipation in the 2022 World Cup Final: **Argentina Reads France** and **France Reads Argentina** directly from `match_10517_final_features.npz`.


In [ ]:
print('Recalculating cross-team anticipation networks directly from .npz...')
t0 = time.time()

arg_cross_roster = {
    'ARG_3':'Tagliafico', 'ARG_7':'De Paul', 'ARG_9':'Alvarez', 'ARG_10':'Messi',
    'ARG_13':'Romero', 'ARG_19':'Otamendi', 'ARG_20':'Mac Allister', 'ARG_21':'Dybala',
    'ARG_25':'L.Martinez', 'ARG_26':'Molina'
}

fra_cross_roster = {
    'FRA_4':'Varane', 'FRA_7':'Griezmann', 'FRA_8':'Tchouameni', 'FRA_9':'Giroud',
    'FRA_10':'Mbappe', 'FRA_11':'Dembele', 'FRA_12':'KoloMuani', 'FRA_14':'Rabiot',
    'FRA_18':'Upamecano', 'FRA_20':'Coman', 'FRA_22':'THernandez', 'FRA_25':'Camavinga', 'FRA_26':'M.Thuram'
}

# (B) Cross-Team: Argentina Reads France (ARG Leader -> FRA Follower)
arg_reads_fra = run_window_anticipation_counts(
    'data/extracted_features/match_10517_final_features.npz',
    arg_cross_roster, fra_cross_roster
)

# (C) Cross-Team: France Reads Argentina (FRA Leader -> ARG Follower)
fra_reads_arg = run_window_anticipation_counts(
    'data/extracted_features/match_10517_final_features.npz',
    fra_cross_roster, arg_cross_roster
)

print(f'Cross-team networks recalculated in {time.time()-t0:.1f}s!')


## 7. Render Figure 2: Intra-Team vs. Cross-Team Anticipation Networks
Plotting all 3 panels directly from the newly calculated results:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(28, 9.5))
fig.suptitle('FIFA World Cup Final 2022 : In-Team vs. Cross-Team Anticipation Event Counts (15s Windows)\n'
             'Total Discrete Moments Leader Triggered Follower (Controlling for Ball Motion, No Masks)',
             fontsize=16, fontweight='bold', y=0.98)

# Panel A: France In-Team
sns.heatmap(fra_counts, ax=axes[0], cmap='YlGnBu', annot=True, fmt='d', linewidths=0.3,
            mask=None, cbar_kws={'label': 'Event Count (15s Windows)'}, vmin=0)
axes[0].set_title('(A) France In-Team Chemistry\n(Leader: France -> Follower: France)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Follower (France)', fontsize=10); axes[0].set_ylabel('Leader (France)', fontsize=10)
axes[0].tick_params(axis='x', rotation=45); axes[0].tick_params(axis='y', rotation=0)

# Panel B: Argentina Reads France
sns.heatmap(arg_reads_fra, ax=axes[1], cmap='YlGnBu', annot=True, fmt='d', linewidths=0.3,
            mask=None, cbar_kws={'label': 'Event Count (15s Windows)'}, vmin=0)
axes[1].set_title('(B) Cross-Team: Argentina Reads France\n(Leader: Argentina Defender/Midfielder -> Follower: France Attacker)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Follower (France)', fontsize=10); axes[1].set_ylabel('Leader (Argentina)', fontsize=10)
axes[1].tick_params(axis='x', rotation=45); axes[1].tick_params(axis='y', rotation=0)

# Panel C: France Reads Argentina
sns.heatmap(fra_reads_arg, ax=axes[2], cmap='YlGnBu', annot=True, fmt='d', linewidths=0.3,
            mask=None, cbar_kws={'label': 'Event Count (15s Windows)'}, vmin=0)
axes[2].set_title('(C) Cross-Team: France Reads Argentina\n(Leader: France Defender/Midfielder -> Follower: Argentina Attacker)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Follower (Argentina)', fontsize=10); axes[2].set_ylabel('Leader (France)', fontsize=10)
axes[2].tick_params(axis='x', rotation=45); axes[2].tick_params(axis='y', rotation=0)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


## 8. Verifying Abstract Key Statistics Directly from Recalculated Arrays


In [ ]:
print('=== VERIFICATION OF PAPER STATISTICS FROM SCRATCH ===')
print(f"France:    Tchouaméni -> Rabiot: {fra_counts.loc['Tchouameni', 'Rabiot']} | Rabiot -> Tchouaméni: {fra_counts.loc['Rabiot', 'Tchouameni']}")
print(f"Argentina: Alvarez -> Messi: {arg_counts.loc['Alvarez', 'Messi']} | Mac Allister <-> Messi: {arg_counts.loc['Mac Allister', 'Messi']}")
print(f"Morocco:   Ziyech -> Hakimi: {mar_counts.loc['Ziyech', 'Hakimi']} | Hakimi -> Ziyech: {mar_counts.loc['Hakimi', 'Ziyech']}")
print(f"Serbia:    Tadić -> Mitrović: {srb_counts.loc['Tadic', 'A.Mitrovic']} | Mitrović -> Tadić: {srb_counts.loc['A.Mitrovic', 'Tadic']}")
print(f"Final:     Mac Allister -> Tchouaméni: {arg_reads_fra.loc['Mac Allister', 'Tchouameni']}")
print(f"Final:     Tchouaméni -> Mac Allister: {fra_reads_arg.loc['Tchouameni', 'Mac Allister']} | Messi: {fra_reads_arg.loc['Tchouameni', 'Messi']} | Álvarez: {fra_reads_arg.loc['Tchouameni', 'Alvarez']}")
